# Assignment 2 — Shopper Purchase Intention

BITS Virtual Lab runbook. This notebook trains five classifiers on the UCI Online Shoppers Purchasing Intention dataset and prints the six required metrics on a stratified held-out test split.

Run all cells from the `Assignment2` project folder (or set `PROJECT_ROOT` below).

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "train_models.py").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from train_models import (
    FEATURE_COLUMNS,
    MODEL_SPECS,
    TARGET_COLUMN,
    build_model_pipeline,
    load_session_table,
    probability_for_purchase,
    score_predictions,
)
from sklearn.model_selection import train_test_split

print("Project root:", PROJECT_ROOT)
print("Feature count:", len(FEATURE_COLUMNS))

In [ ]:
sessions = load_session_table()
print("Rows:", len(sessions))
print("Columns:", list(sessions.columns))
print("Purchase rate:\n", sessions[TARGET_COLUMN].value_counts(normalize=True).round(4))
sessions.head()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    sessions[FEATURE_COLUMNS],
    sessions[TARGET_COLUMN],
    test_size=0.20,
    stratify=sessions[TARGET_COLUMN],
    random_state=42,
)
print("Train sessions:", len(x_train), "| Test sessions:", len(x_test))

In [ ]:
metric_rows = {}
fitted = {}
for model_name, (_filename, estimator) in MODEL_SPECS.items():
    pipeline = build_model_pipeline(estimator)
    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)
    y_score = probability_for_purchase(pipeline, x_test)
    metric_rows[model_name] = score_predictions(y_test, y_pred, y_score)
    fitted[model_name] = (pipeline, y_pred)
    print(model_name, metric_rows[model_name])

comparison = pd.DataFrame(metric_rows).T
comparison

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

winner_name = comparison["MCC"].idxmax()
_, winner_pred = fitted[winner_name]
print("Highest MCC model:", winner_name)
print(classification_report(y_test, winner_pred, target_names=["No purchase", "Purchase"]))

matrix = confusion_matrix(y_test, winner_pred, labels=[False, True])
plt.figure(figsize=(5, 4))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="YlGnBu",
    xticklabels=["No purchase", "Purchase"],
    yticklabels=["No purchase", "Purchase"],
)
plt.title(f"Confusion matrix — {winner_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()